In [10]:
import pandas as pd

DATA_PATH = "../data/raw/diabetic_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (101766, 50)


In [11]:
# Create the 30-day readmission target

df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

print("Target distribution:")
print(df["readmitted_30"].value_counts())

print("\nTarget proportions:")
print(df["readmitted_30"].value_counts(normalize=True))

Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64

Target proportions:
readmitted_30
0    0.888401
1    0.111599
Name: proportion, dtype: float64


In [12]:
# Identify patient ID and target

PATIENT_ID = "patient_nbr"
TARGET = "readmitted_30"

print("Patient ID:", PATIENT_ID)
print("Target:", TARGET)

print("Unique patients:", df[PATIENT_ID].nunique())
print("Total encounters:", len(df))

Patient ID: patient_nbr
Target: readmitted_30
Unique patients: 71518
Total encounters: 101766


In [13]:
# Create unique patient list

patients = df[PATIENT_ID].drop_duplicates()

print("Unique patients:", len(patients))
print("First 10 patient IDs:")
print(patients.head(10).tolist())

Unique patients: 71518
First 10 patient IDs:
[8222157, 55629189, 86047875, 82442376, 42519267, 82637451, 84259809, 114882984, 48330783, 63555939]


In [14]:
from sklearn.model_selection import train_test_split

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

print("Training patients:", len(train_patients))
print("Test patients:", len(test_patients))

Training patients: 57214
Test patients: 14304


In [15]:
# Create patient-level train and test datasets

train_df = df[df[PATIENT_ID].isin(train_patients)].copy()
test_df = df[df[PATIENT_ID].isin(test_patients)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Train patients:", train_df[PATIENT_ID].nunique())
print("Test patients:", test_df[PATIENT_ID].nunique())

Train shape: (81477, 51)
Test shape: (20289, 51)
Train patients: 57214
Test patients: 14304


In [16]:
# Check for patient overlap between train and test

train_patient_ids = set(train_df[PATIENT_ID])
test_patient_ids = set(test_df[PATIENT_ID])

overlap = train_patient_ids.intersection(test_patient_ids)

print("Patients in both train and test:", len(overlap))

if len(overlap) == 0:
    print("PASS: No patient overlap detected.")
else:
    print("WARNING: Patient overlap detected!")

Patients in both train and test: 0
PASS: No patient overlap detected.


In [17]:
# Compare target distribution between train and test

print("Train target distribution:")
print(train_df[TARGET].value_counts())

print("\nTrain target proportions:")
print(train_df[TARGET].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df[TARGET].value_counts())

print("\nTest target proportions:")
print(test_df[TARGET].value_counts(normalize=True))

Train target distribution:
readmitted_30
0    72289
1     9188
Name: count, dtype: int64

Train target proportions:
readmitted_30
0    0.887232
1    0.112768
Name: proportion, dtype: float64

Test target distribution:
readmitted_30
0    18120
1     2169
Name: count, dtype: int64

Test target proportions:
readmitted_30
0    0.893095
1    0.106905
Name: proportion, dtype: float64


In [18]:
# Verify train + test covers all encounters

total_split_rows = len(train_df) + len(test_df)

print("Original encounters:", len(df))
print("Train encounters:", len(train_df))
print("Test encounters:", len(test_df))
print("Train + Test:", total_split_rows)

if total_split_rows == len(df):
    print("PASS: All encounters are included.")
else:
    print("WARNING: Encounter count mismatch.")

Original encounters: 101766
Train encounters: 81477
Test encounters: 20289
Train + Test: 101766
PASS: All encounters are included.


## Patient-Level Train/Test Split

The dataset was split at the patient level to prevent information leakage between training and test data.

### Split configuration
- Total unique patients: 71,518
- Training patients: 57,214
- Test patients: 14,304
- Split ratio: 80% training / 20% test
- Random state: 42

### Encounter distribution
- Total encounters: 101,766
- Training encounters: 81,477
- Test encounters: 20,289

### Leakage verification
- Patients appearing in both train and test: 0
- All encounters included after splitting: Yes

### Target distribution
- Training positive rate: 11.28%
- Test positive rate: 10.69%

The patient-level split ensures that encounters from the same patient do not appear in both training and test datasets.